# Holdie — אימון מסווג אודיו ב-Google Colab

המחברת הזו מבצעת מקצה-לקצה: קלון של הריפו, חיתוך האודיו לחתיכות של 5 שניות, אימון על GPU, ואז שמירת ה-checkpoints חזרה ל-Drive.

**לפני שמתחילים:**
1. `Runtime → Change runtime type → GPU` (T4 חינמי מספיק להתחלה; A100/L4 ירוצו מהר יותר).
2. סדרו את האודיו הגולמי ב-Google Drive במבנה הבא:

```
MyDrive/holdie-audio/raw/
├── human/       ← הקלטות של נציגים אנושיים
├── ivr/         ← תפריטי IVR (כולל רב-לשוניים)
├── music/       ← מוזיקת המתנה
└── recording/   ← הקלטות / תא קולי
```

**הערה:** אודיו IVR עשוי להכיל משפטים בשפות שונות (עברית, אנגלית, רוסית וכו') — זה בסדר. המודל מזהה מאפיינים אקוסטיים (קול סינתטי, קצב, טון) ולא תוכן שפתי, אז אין צורך למיין לפי שפה.

## 1. בדיקת ה-GPU

In [ ]:
!nvidia-smi

## 2. קלון של הריפו והתקנת תלויות

In [ ]:
REPO_URL = "https://github.com/YossiYad/research.git"
BRANCH = "main"

import os, subprocess
if not os.path.exists("/content/research"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/research"], check=True)
else:
    subprocess.run(["git", "-C", "/content/research", "pull", "--ff-only"], check=True)

%cd /content/research

In [ ]:
!pip install -q -r requirements.txt

## 3. חיבור Google Drive

ה-Drive נטען ל-`/content/drive`. אם זו הפעם הראשונה, ייפתח חלון אישור.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. הגדרת נתיבים

אם בחרתם מבנה תיקיות אחר ב-Drive — עדכנו את `DRIVE_RAW` כאן.

In [ ]:
from pathlib import Path

DRIVE_RAW = Path("/content/drive/MyDrive/holdie-audio/raw")
DRIVE_OUT = Path("/content/drive/MyDrive/holdie-audio/checkpoints")

PROJECT = Path("/content/research")
LOCAL_RAW = PROJECT / "dataset" / "raw"
LOCAL_PROCESSED = PROJECT / "dataset" / "processed"
METADATA = PROJECT / "dataset" / "metadata.csv"
CHECKPOINTS = PROJECT / "checkpoints"

assert DRIVE_RAW.exists(), f"לא נמצא {DRIVE_RAW} — ודאו שהעליתם את האודיו ל-Drive במבנה הנכון"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("DRIVE_RAW:", DRIVE_RAW)
print("DRIVE_OUT:", DRIVE_OUT)

## 5. חיתוך האודיו (אוטומטי לכל קטגוריה)

התא הזה סורק את `DRIVE_RAW/{class}/`, חותך כל תיקייה לחתיכות של 5 שניות, ושומר ב-`dataset/processed/{class}/`.

In [ ]:
import subprocess, sys

VALID_CLASSES = ["human", "ivr", "music", "recording"]

# מחיקה של metadata.csv ישן כדי שהריצה תהיה דטרמיניסטית
if METADATA.exists():
    print(f"מוחק {METADATA} ישן ובונה מחדש")
    METADATA.unlink()

for cls in VALID_CLASSES:
    cls_dir = DRIVE_RAW / cls
    if not cls_dir.exists():
        print(f"⏭ {cls}: אין תיקייה ב-Drive, מדלג")
        continue

    print(f"\n=== {cls} ===")
    subprocess.run([
        sys.executable, "scripts/chop_audio.py",
        "--input", str(cls_dir),
        "--class", cls,
    ], check=True)

In [ ]:
# סיכום מהיר של מה שהתקבל
import sys
sys.path.insert(0, str(PROJECT / "scripts"))
from dataset import load_metadata, class_distribution

meta = load_metadata(METADATA)
print(f"סה\"כ חתיכות: {len(meta)}")
print("לפי קטגוריה:", class_distribution(meta))

## 6. אימון

בחירת מודל הבסיס:
- **`facebook/wav2vec2-base`** — 95M פרמטרים, מהיר, אומן על אנגלית. מתאים להתחלה ולסט נתונים קטן.
- **`facebook/wav2vec2-xls-r-300m`** — 300M פרמטרים, רב-לשוני (53 שפות כולל עברית). **ברירת מחדל.**
- **`imvladikon/wav2vec2-xls-r-300m-hebrew`** — מותאם לעברית, מומלץ כשרוב המאגר בעברית.

פרמטרים חדשים:
- **`UNFREEZE_LAYERS`** — מספר שכבות encoder עליונות לשחרר (ברירת מחדל: 2). מאפשר fine-tuning חלקי עם LR נמוך.
- **`ENCODER_LR`** — קצב למידה לשכבות המשוחררות (ברירת מחדל: 1e-5, נמוך פי 10 מה-LR הרגיל).
- **`WARMUP_EPOCHS`** — אפוקי warmup לינארי לפני cosine annealing.
- Early stopping מבוסס כעת על **macro F1** במקום accuracy.

In [ ]:
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-4
UNFREEZE_LAYERS = 2       # שכבות encoder עליונות לשחרור (0 = הכל קפוא)
ENCODER_LR = 1e-5         # LR נמוך לשכבות encoder משוחררות
WARMUP_EPOCHS = 2         # אפוקי warmup לינארי

!python scripts/train.py \
  --base-model {BASE_MODEL} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR} \
  --unfreeze-layers {UNFREEZE_LAYERS} \
  --encoder-lr {ENCODER_LR} \
  --warmup-epochs {WARMUP_EPOCHS}

## 7. ויזואליזציה

מציג גרף הפסד/דיוק לאורך האפוקים ומטריצת בלבול.

In [ ]:
!python scripts/visualize.py

## 7.5 בדיקה על קובץ בודד (אופציונלי)

אפשר לבדוק את המודל על קובץ אודיו ספציפי מה-Drive:

In [ ]:
import shutil
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
dest = DRIVE_OUT / f"run-{stamp}"
dest.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt", "history.json", "run_config.json",
             "training_curves.png", "confusion_matrix.png",
             "class_distribution.png", "per_class_metrics.png"):
    src = CHECKPOINTS / name
    if src.exists():
        shutil.copy2(src, dest / name)
        print(f"✓ {name} → {dest / name}")

# גם metadata.csv לתיעוד מלא של המאגר שהמודל אומן עליו
if METADATA.exists():
    shutil.copy2(METADATA, dest / "metadata.csv")
    print(f"✓ metadata.csv → {dest / 'metadata.csv'}")

print(f"\nכל הריצה נשמרה ב: {dest}")

## 8. שמירה חזרה ל-Drive

ה-checkpoints נשמרים גם מקומית (תיאבדו עם סשן Colab) — חשוב להעתיק אותם ל-Drive.

In [ ]:
import shutil
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
dest = DRIVE_OUT / f"run-{stamp}"
dest.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt", "history.json", "run_config.json",
             "training_curves.png", "confusion_matrix.png",
             "class_distribution.png", "per_class_metrics.png"):
    src = CHECKPOINTS / name
    if src.exists():
        shutil.copy2(src, dest / name)
        print(f"✓ {name} → {dest / name}")

# גם metadata.csv לתיעוד מלא של המאגר שהמודל אומן עליו
if METADATA.exists():
    shutil.copy2(METADATA, dest / "metadata.csv")
    print(f"✓ metadata.csv → {dest / 'metadata.csv'}")

print(f"\nכל הריצה נשמרה ב: {dest}")